In [3]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

/Users/macuser/projects/ufc_prediction/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
def make_soup(url: str) -> BeautifulSoup:
    source_code = requests.get(url, allow_redirects=False)
    plain_text = source_code.text.encode("ascii", "replace")
    return BeautifulSoup(plain_text, "html.parser")

def get_all_event_details(soup_obj):

    all_event_links = []

    for link in soup_obj.find_all("td", {"class": "b-statistics__table-col"}):
                for href in link.find_all("a"):
                    event_link = href.get("href")
                    all_event_links.append(event_link)

    return all_event_links

def get_all_fight_details(event_soup_obj):
      
    fight_details_links = []

    for fight_detail in event_soup_obj.find_all("tr", {"class": "b-fight-details__table-row b-fight-details__table-row__hover js-fight-details-click"}):
        href = fight_detail.get("data-link")
        fight_details_links.append(href)

    return fight_details_links


In [5]:
all_events_url="http://ufcstats.com/statistics/events/completed?page=all"

test_soup = make_soup(all_events_url)

all_event_links = get_all_event_details(test_soup)

test_event = all_event_links[1]

test_event_soup = make_soup(test_event)

fight_links = get_all_fight_details(test_event_soup)
fight_links[:5]

['http://ufcstats.com/fight-details/d69bd8e53ec858dc',
 'http://ufcstats.com/fight-details/721642e7de112e9e',
 'http://ufcstats.com/fight-details/ad487d1c6141d90a',
 'http://ufcstats.com/fight-details/fef80c98b2fba70c',
 'http://ufcstats.com/fight-details/c27bec45d5a29a2c']

In [6]:
fight_soup = make_soup(fight_links[0])


In [7]:
tables = fight_soup.findAll("tbody")
test_table = tables[0]


/var/folders/ft/jgrps1md2_30cc3nx_zr573r0000gn/T/ipykernel_45786/459022420.py:1: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  tables = fight_soup.findAll("tbody")


In [8]:
tables = fight_soup.findAll("tbody")
totals_table = tables[0]
totals_per_round_table = tables[1]
significant_strikes_table = tables[2]
significant_strikes_per_round_table = tables[3]


/var/folders/ft/jgrps1md2_30cc3nx_zr573r0000gn/T/ipykernel_45786/4181895494.py:1: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  tables = fight_soup.findAll("tbody")


In [9]:
type(totals_per_round_table.find('tr'))
# type(totals_per_round_table.find_all('tr'))

bs4.element.Tag

In [10]:
len(totals_per_round_table.find_all('tr'))

2

In [25]:
test_sig_str_data = significant_strikes_table.find('tr').find_all('td')
for i in test_sig_str_data:
    print(i.text.replace("\n    ",""))



Renato Moicano 


Chris Duncan 



  21 of 36
  27 of 62


  58%
  43%


  16 of 31
  8 of 39


  4 of 4
  5 of 7


  1 of 1
  14 of 16


  20 of 34
  25 of 60


  1 of 1
  2 of 2


  0 of 1
  0 of 0



In [26]:
totals_table_structure = {0: ['fighters', '\n'],
                          1: ['knockdowns', '\n    '],
                          2: ['significant strikes', '\n    '],
                          3: ['significant strike percentage', '\n    '],
                          4: ['total strike', '\n    '],
                          5: ['takedowns', '\n    '],
                          6: ['takedown percentage', '\n    '],
                          7: ['submission attempts', '\n    '],
                          8: ['reversals', '\n    '],
                          9: ['control', '\n    ']}

sig_str_table_structure = {0: ['fighters', '\n'],
                          1: ['significant strikes', '\n    '],
                          2: ['significant strike percentage', '\n    '],
                          3: ['head', '\n    '],
                          4: ['body', '\n    '],
                          5: ['leg', '\n    '],
                          6: ['distance', '\n    '],
                          7: ['clinch', '\n    '],
                          8: ['ground', '\n    ']}


In [13]:
## get all of the table data and initialize the data dictionary
totals_dat = totals_table.find_all('td')
test_total_dict = {}

for i in range(len(totals_dat)):
    ## remove new lines and tabs for easier formatting
    table_row_data_clean = totals_dat[i].text.replace("\n\n", "").replace("      ","").replace("\n    \n","")

    ## fill in the dictioanry for the corresponding structure
    test_total_dict[totals_table_structure[i][0]] = table_row_data_clean.split(totals_table_structure[i][1])

print(test_total_dict)

{'fighters': ['Renato Moicano ', 'Chris Duncan '], 'knockdowns': ['0', '0'], 'significant strikes': ['21 of 36', '27 of 62'], 'significant strike percentage': ['58%', '43%'], 'total strike': ['62 of 84', '30 of 66'], 'takedowns': ['1 of 4', '0 of 2'], 'takedown percentage': ['25%', '0%'], 'submission attempts': ['1', '0'], 'reversals': ['0', '0'], 'control': ['3:16', '0:01']}


In [37]:
# totals_table
def parse_total_table(table, table_structure):

    ## get all of the table data and initialize the data dictionary
    table_data = table.find_all('td')
    table_dict = {}

    for i in range(len(table_data)):
        ## remove new lines and tabs for easier formatting
        table_data_clean = table_data[i].text.replace("\n\n", "").replace("      ","").replace("\n    \n","")

        ## fill in the dictioanry for the corresponding structure
        table_dict[table_structure[i][0]] = table_data_clean.split(table_structure[i][1])

    return table_dict

# parse_total_table(totals_table,totals_table_structure)
parse_total_table(significant_strikes_table,sig_str_table_structure)



{'fighters': ['Renato Moicano ', 'Chris Duncan '],
 'significant strikes': ['21 of 36', '27 of 62'],
 'significant strike percentage': ['58%', '43%'],
 'head': ['16 of 31', '8 of 39'],
 'body': ['4 of 4', '5 of 7'],
 'leg': ['1 of 1', '14 of 16'],
 'distance': ['20 of 34', '25 of 60'],
 'clinch': ['1 of 1', '2 of 2'],
 'ground': ['0 of 1', '0 of 0']}

In [ ]:
def parse_per_round_table(per_round_table, per_round_table_structure):

    ## Save all table headers and table rows:
    headers_and_rows = per_round_table.find_all(['th','tr'])

    ## Initialize the stats_per_round_dict and round unumber object
    stats_per_round_dict = {}
    round = None

    for i in headers_and_rows:

        ## if the soup_item is a table header, set our round variable to it
        if i.name == 'th':
            round_num = i.text.replace("\n              ", "").replace("\n            ","")
            # print(round_num)

        ## if the soup_item is a table row, parse the table row data and assign it to the current round
        if i.name == 'tr':

             stats_per_round_dict[round_num] = extract_data_from_table(i, per_round_table_structure)

    return stats_per_round_dict

# parse_per_round_table(totals_per_round_table,totals_table_structure)
parse_per_round_table(significant_strikes_per_round_table,sig_str_table_structure)


{'Round 1': {'fighters': ['Renato Moicano ', 'Chris Duncan '],
  'significant strikes': ['14 of 20', '17 of 38'],
  'significant strike percentage': ['70%', '44%'],
  'head': ['10 of 16', '3 of 22'],
  'body': ['3 of 3', '3 of 3'],
  'leg': ['1 of 1', '11 of 13'],
  'distance': ['13 of 19', '15 of 36'],
  'clinch': ['1 of 1', '2 of 2'],
  'ground': ['0 of 0', '0 of 0']},
 'Round 2': {'fighters': ['Renato Moicano ', 'Chris Duncan '],
  'significant strikes': ['7 of 16', '10 of 24'],
  'significant strike percentage': ['43%', '41%'],
  'head': ['6 of 15', '5 of 17'],
  'body': ['1 of 1', '2 of 4'],
  'leg': ['0 of 0', '3 of 3'],
  'distance': ['7 of 15', '10 of 24'],
  'clinch': ['0 of 0', '0 of 0'],
  'ground': ['0 of 1', '0 of 0']}}

In [ ]:

## Save all table headers and table rows:
headers_and_rows = totals_per_round_table.find_all(['th','tr'])

## Initialize the stats_per_round_dict and round unumber object
stats_per_round_dict = {}
round = None

for i in headers_and_rows:

    ## if the soup_item is a table header, set our round variable to it
    if i.name == 'th':
        round_num = i.text.replace("\n              ", "").replace("\n            ","")
        # print(round_num)

    ## if the soup_item is a table row, parse the table row data and assign it to the current round
    if i.name == 'tr':
        round_row_data = i.find_all('td')

        ind_round_dict = {}

        for i in range(len(round_row_data)):

            ## remove new lines and tabs for easier formatting
            round_row_data_clean = round_row_data[i].text.replace("\n\n", "").replace("      ","").replace("\n    \n","")

            ## fill in the individual_round_dictioanry for the corresponding structure
            ind_round_dict[totals_table_structure[i][0]] = round_row_data_clean.split(totals_table_structure[i][1])

        stats_per_round_dict[round_num] = ind_round_dict
print(stats_per_round_dict.keys())
print(stats_per_round_dict)




dict_keys(['Round 1', 'Round 2'])
{'Round 1': {'fighters': ['Israel Adesanya ', 'Joe Pyfer '], 'knockdowns': ['0', '0'], 'significant strikes': ['24 of 42', '9 of 23'], 'significant strike percentage': ['57%', '39%'], 'total strike': ['37 of 56', '9 of 23'], 'takedowns': ['0 of 0', '1 of 4'], 'takedown percentage': ['---', '25%'], 'submission attempts': ['0', '0'], 'reversals': ['0', '0'], 'control': ['0:00', '1:01']}, 'Round 2': {'fighters': ['Israel Adesanya ', 'Joe Pyfer '], 'knockdowns': ['0', '0'], 'significant strikes': ['18 of 33', '27 of 47'], 'significant strike percentage': ['54%', '57%'], 'total strike': ['21 of 36', '43 of 67'], 'takedowns': ['0 of 2', '1 of 4'], 'takedown percentage': ['0%', '25%'], 'submission attempts': ['0', '0'], 'reversals': ['0', '0'], 'control': ['0:00', '1:43']}}


In [38]:
totals_table_structure = {0: ['fighters', '\n'],
                          1: ['knockdowns', '\n    '],
                          2: ['significant strikes', '\n    '],
                          3: ['significant strike percentage', '\n    '],
                          4: ['total strike', '\n    '],
                          5: ['takedowns', '\n    '],
                          6: ['takedown percentage', '\n    '],
                          7: ['submission attempts', '\n    '],
                          8: ['reversals', '\n    '],
                          9: ['control', '\n    ']}

sig_str_table_structure = {0: ['fighters', '\n'],
                          1: ['significant strikes', '\n    '],
                          2: ['significant strike percentage', '\n    '],
                          3: ['head', '\n    '],
                          4: ['body', '\n    '],
                          5: ['leg', '\n    '],
                          6: ['distance', '\n    '],
                          7: ['clinch', '\n    '],
                          8: ['ground', '\n    ']}

def parse_agg_table(table, table_structure):

    ## get all of the table data and initialize the data dictionary
    table_data = table.find_all('td')
    table_dict = {}

    for i in range(len(table_data)):
        ## remove new lines and tabs for easier formatting
        table_data_clean = table_data[i].text.replace("\n\n", "").replace("      ","").replace("\n    \n","")

        ## fill in the dictioanry for the corresponding structure
        table_dict[table_structure[i][0]] = table_data_clean.split(table_structure[i][1])

    return table_dict

def parse_per_round_table(per_round_table, per_round_table_structure):

    ## Save all table headers and table rows:
    headers_and_rows = per_round_table.find_all(['th','tr'])

    ## Initialize the stats_per_round_dict and round unumber object
    stats_per_round_dict = {}
    round = None

    for i in headers_and_rows:

        ## if the soup_item is a table header, set our round variable to it
        if i.name == 'th':
            round_num = i.text.replace("\n              ", "").replace("\n            ","")
            # print(round_num)

        ## if the soup_item is a table row, parse the table row data and assign it to the current round
        if i.name == 'tr':

             stats_per_round_dict[round_num] = parse_agg_table(i, per_round_table_structure)

    return stats_per_round_dict

In [41]:
#function: 
#   seperate fight data into different sections  -- DONE
#       for each fight section:
#           extract the correct amount of tables (1 or 2) -- DONE
#   for each extracted table:
#       get the information using a pre-made dictioanry.  -- DONE
#       Append the resulting row to a csv

def extract_fight_data(fight_soup, totals_table_structure, sig_str_table_structure):

    fight_tables = fight_soup.findAll("tbody")
    fight_data = {}

    ## seperate fight data into different sections 
    totals_table = fight_tables[0]
    totals_per_round_tables = fight_tables[1]
    sig_str_table = fight_tables[2]
    sig_str_per_round_tables = fight_tables[3]

    ## combine per_round tables into a single table
    fight_data['fight_totals'] = parse_agg_table(totals_table, totals_table_structure)
    fight_data['totals_per_round'] = parse_per_round_table(totals_per_round_tables, totals_table_structure)

    ## sig_str table
    fight_data['sig_str'] = parse_agg_table(sig_str_table, sig_str_table_structure)
    fight_data['sig_str_per_round'] = parse_per_round_table(sig_str_per_round_tables, sig_str_table_structure)

    return fight_data

test_fight_stats = extract_fight_data(fight_soup, totals_table_structure, sig_str_table_structure)


/var/folders/ft/jgrps1md2_30cc3nx_zr573r0000gn/T/ipykernel_45786/2665430960.py:11: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  fight_tables = fight_soup.findAll("tbody")


In [42]:
test_fight_stats

{'fight_totals': {'fighters': ['Renato Moicano ', 'Chris Duncan '],
  'knockdowns': ['0', '0'],
  'significant strikes': ['21 of 36', '27 of 62'],
  'significant strike percentage': ['58%', '43%'],
  'total strike': ['62 of 84', '30 of 66'],
  'takedowns': ['1 of 4', '0 of 2'],
  'takedown percentage': ['25%', '0%'],
  'submission attempts': ['1', '0'],
  'reversals': ['0', '0'],
  'control': ['3:16', '0:01']},
 'totals_per_round': {'Round 1': {'fighters': ['Renato Moicano ',
    'Chris Duncan '],
   'knockdowns': ['0', '0'],
   'significant strikes': ['14 of 20', '17 of 38'],
   'significant strike percentage': ['70%', '44%'],
   'total strike': ['16 of 22', '19 of 41'],
   'takedowns': ['0 of 2', '0 of 1'],
   'takedown percentage': ['0%', '0%'],
   'submission attempts': ['0', '0'],
   'reversals': ['0', '0'],
   'control': ['1:13', '0:01']},
  'Round 2': {'fighters': ['Renato Moicano ', 'Chris Duncan '],
   'knockdowns': ['0', '0'],
   'significant strikes': ['7 of 16', '10 of 24']